In [1]:
# import any required  packages here

import mat73
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import pandas as pd

import sklearn as sk
from statistics import mean
from sklearn.linear_model import LinearRegression
from scipy.signal import butter, lfilter, iirnotch

In [2]:
# subject demographic information from the dataset paper
# there are a total of 10 participants

mass = np.array([63.49, 63.49, 71.2, 68.03, 68.03, 68.03, 95.24, 65.76, 68.93, 58.05]) # [kg]
height = np.array([1.85, 1.77, 1.73, 1.68, 1.83, 1.85, 1.85, 1.85, 1.73, 1.63]) # [m]
age = np.array([30, 25, 24, 25, 24, 25, 24, 33, 37, 27]) # [years]
sex = np.array([0, 0, 1, 0, 0, 0, 0, 0, 0, 1]) # 0 = male, 1 = female

In [3]:
# creating a dictionary for all of the activities
# more a convenience than anything

thisdict = {
    "activity": ["Walking", "Incline", "Backwards", "Running", "Cycling", "Stairs", "Standing", "Sitting"],
    "Wspeed": ["0.6", "0.9", "1.2"],
    "Iangle": ["4", "9"],
    "Ispeed": ["0.6", "1.2"],
    "Bspeed": ["0.4", "0.7", "1"],
    "Rspeed": ["1.2", "1.8", "2.2", "2.7"],
    "Cinten": ["1", "3", "5"],
    "Cspeed": ["70", "100"],
    "StairI": ["60"]
}

def activityCode(code,thisdict):
    switcher = {
        1: [thisdict["activity"][0], "at", thisdict["Wspeed"][0], "m/s"],
        2: [thisdict["activity"][0], "at", thisdict["Wspeed"][1], "m/s"],
        3: [thisdict["activity"][0], "at", thisdict["Wspeed"][2], "m/s"],
        4: [thisdict["activity"][1], "at", thisdict["Ispeed"][0], "m/s", "at", thisdict["Iangle"][0], "deg"],
        5: [thisdict["activity"][1], "at", thisdict["Ispeed"][1], "m/s", "at", thisdict["Iangle"][0], "deg"],
        6: [thisdict["activity"][1], "at", thisdict["Ispeed"][0], "m/s", "at", thisdict["Iangle"][1], "deg"],
        7: [thisdict["activity"][1], "at", thisdict["Ispeed"][1], "m/s", "at", thisdict["Iangle"][1], "deg"],
        8: [thisdict["activity"][2], "at", thisdict["Bspeed"][0], "m/s"],
        9: [thisdict["activity"][2], "at", thisdict["Bspeed"][1], "m/s"],
        10: [thisdict["activity"][2], "at", thisdict["Bspeed"][2], "m/s"],
        11: [thisdict["activity"][0], "at", thisdict["Rspeed"][0], "m/s"],
        12: [thisdict["activity"][3], "at", thisdict["Rspeed"][1], "m/s"],
        13: [thisdict["activity"][3], "at", thisdict["Rspeed"][2], "m/s"],
        14: [thisdict["activity"][3], "at", thisdict["Rspeed"][3], "m/s"],
        15: [thisdict["activity"][4], "at", thisdict["Cspeed"][0], "rpm", "at Resistance", thisdict["Cinten"][0]],
        16: [thisdict["activity"][4], "at", thisdict["Cspeed"][0], "rpm", "at Resistance", thisdict["Cinten"][1]],
        17: [thisdict["activity"][4], "at", thisdict["Cspeed"][0], "rpm", "at Resistance", thisdict["Cinten"][2]],
        18: [thisdict["activity"][4], "at", thisdict["Cspeed"][1], "rpm", "at Resistance", thisdict["Cinten"][0]],
        19: [thisdict["activity"][5], "at", thisdict["StairI"][0], "W"],
        20: [thisdict["activity"][5], "at", thisdict["StairI"][0], "W"],
        21: [thisdict["activity"][5], "at", thisdict["StairI"][0], "W"],
        22: thisdict["activity"][6],
        23: thisdict["activity"][7]
    }
    
    my_list = switcher.get(code, "Invalid code")
    if my_list=="Invalid code":
        my_string = my_list
    elif code < 22:
        my_string = " ".join(my_list)
    else:
        my_string = my_list

    return my_string
    
print(activityCode(1,thisdict))

Walking at 0.6 m/s


56 APDM_Accel signals

- 0: time
- 1: activity code
- 2-10: Waist signals
- 11-19: Chest signals
- 20-28: Left Ankle signals
- 29-37: Right Ankle signals
- 38-46: Left Foot signals
- 47-55: Right Foot signals

Within the APDM_Accel sensor signals
- 0-2: acceleration
- 3-5: angular velocity
- 6-8: magnetic field

In [4]:
# indices for different elements within APDM_Accel

time = 0 # same index for all
code = 1 # same index for all
waist = [2, 11]
chest = [11, 20]
Lshnk = [20, 29]
Rshnk = [29, 38]
Lfoot = [38, 47]
Rfoot = [47, 56]

acc = [0, 3]
ang = [3, 6]
mag = [6, 9]

18 EMG signals

- 0: time
- 1: activity code
- 2-9: right leg
- 10-17: left leg

Within right/left leg
- [2,10]: gluteus maximus
- [3,11]: rectus femoris
- [4,12]: vastus lateralis
- [5,13]: semitendinosis
- [6,14]: biceps femoris
- [7,15]: medial gastrocnemius
- [8,16]: soleus
- [9,17]: tibialis anterior


In [5]:
# indices for different elements within EMG

Rleg = [2, 10]
Lleg = [10, 18]

glut = 2
rect = 3
vast = 4
semi = 5
bicfem = 6
gastroc = 7
sol = 8
TA = 9

EMGdiff = 8

8 Empatica Accel signals

- 0: time
- 1: activity code
- 2-4: left wrist accel
- 5-7: right wrist accel

In [6]:
# indices for different elements within Empatica_Accel

Lwrist = [2,5]
Rwrist = [5,8]

6 Empatica Physiological signals

- 0: time
- 1: activity code
- 2-3: left wrist
- 4-5: right wrist

Within left/right wrist
- [2,4]: electrodermal activity (EDA)
- [3,5]: skin temperature

In [7]:

# indices for different elements within Empatica_Physio

Lwrist = [2,4]
Rwrist = [4,6]

EDA = 2
temp = 3

PHYdiff = 2

ACC data on
- Waist, Chest, Left Shank, Right Shank, Left Foot, Right Foot, Right Wrist, Left Wrist

ANGVEL data on
- Waist, Chest, Left Shank, Right Shank, Left Foot, Right Foot

9 Metabolics System signals

- 0: time
- 1: activity code
- 2: VO2
- 3: VCO2
- 4: RER
- 5: breath frequency
- 6: minute ventiltation
- 7: oxygen saturation (SpO2)
- 8: heart rate

In [8]:
# indices for different elements within Metabolics_System

vo2 = 2
vco2 = 3
rer = 4
brefr = 5
minVent = 6
spo2 = 7
hr = 8

In [ ]:
# import the data 
# switch folders to be compatible with your computer

mat_fname01 = '/Users/katiebutler/Downloads/HIR Lab/Complete DataSet/Subject01.mat'
mat_fname02 = '/Users/katiebutler/Downloads/HIR Lab/Complete DataSet/Subject02.mat'
mat_fname03 = '/Users/katiebutler/Downloads/HIR Lab/Complete DataSet/Subject03.mat'
mat_fname04 = '/Users/katiebutler/Downloads/HIR Lab/Complete DataSet/Subject04.mat'
mat_fname05 = '/Users/katiebutler/Downloads/HIR Lab/Complete DataSet/Subject05.mat'
mat_fname06 = '/Users/katiebutler/Downloads/HIR Lab/Complete DataSet/Subject06.mat'
mat_fname07 = '/Users/katiebutler/Downloads/HIR Lab/Complete DataSet/Subject07.mat'
mat_fname08 = '/Users/katiebutler/Downloads/HIR Lab/Complete DataSet/Subject08.mat'
mat_fname09 = '/Users/katiebutler/Downloads/HIR Lab/Complete DataSet/Subject09.mat'
mat_fname10 = '/Users/katiebutler/Downloads/HIR Lab/Complete DataSet/Subject10.mat'

mat_contents01 = mat73.loadmat(mat_fname01)
mat_contents02 = mat73.loadmat(mat_fname02)
mat_contents03 = mat73.loadmat(mat_fname03)
mat_contents04 = mat73.loadmat(mat_fname04)
mat_contents05 = mat73.loadmat(mat_fname05)
mat_contents06 = mat73.loadmat(mat_fname06)
mat_contents07 = mat73.loadmat(mat_fname07)
mat_contents08 = mat73.loadmat(mat_fname08)
mat_contents09 = mat73.loadmat(mat_fname09)
mat_contents10 = mat73.loadmat(mat_fname10)

# playing around with the dataframe to make sure I understand how everything is organized here

df1 = pd.DataFrame.from_dict(mat_contents01)
df2 = pd.DataFrame.from_dict(mat_contents02)
df3 = pd.DataFrame.from_dict(mat_contents03)
df4 = pd.DataFrame.from_dict(mat_contents04)
df5 = pd.DataFrame.from_dict(mat_contents05)
df6 = pd.DataFrame.from_dict(mat_contents06)
df7 = pd.DataFrame.from_dict(mat_contents07)
df8 = pd.DataFrame.from_dict(mat_contents08)
df9 = pd.DataFrame.from_dict(mat_contents09)
df10 = pd.DataFrame.from_dict(mat_contents10)

print("Column headers:", list(mat_contents01['Subject01']['Walking']))
 
print(df1['Subject01']['Walking']['Empatica_Physio']['Labels'])
# display(df1)
#print(df1[Subject01])

# plt.plot(df1['Subject01']['Backwards']['APDM_Accel']['Data'][:,0],df1['Subject01']['Backwards']['APDM_Accel']['Data'][:,2:5])
#plt.show()

In [ ]:
result = pd.concat([df1["Subject01"], df2["Subject02"], df3["Subject03"], df4["Subject04"], df5["Subject05"], df6["Subject06"], df7["Subject07"], df8["Subject08"], df9["Subject09"], df10["Subject10"]], keys=["Subject01", "Subject02", "Subject03", "Subject04", "Subject05", "Subject06", "Subject07", "Subject08", "Subject09", "Subject10"])
print(result)

Levels for the datafile
- SubjectXX
    - Backwards, Cycling, Incline, Running, Walking
        - APDM_Accel, EMG, Empatica_Accel, Empatica_Physio, Metabolics_System
            - Data, Labels, SamplingRate

In [ ]:
# calculate energy expenditure according to oxygen uptake and carbon dioxide produced

VO2 = result['Subject01']['Walking']['Metabolics_System']['Data'][:,vo2]
VCO2 = result['Subject01']['Walking']['Metabolics_System']['Data'][:,vco2]

EE = (16.58*VO2 + 4.15*VCO2)/mass[0]
EE_ground = np.zeros(np.size(EE))

In [ ]:
# need to separate out each phase of the test
# ground truth EE is the average of the last three minutes of each phase

ind = result['Subject01']['Walking']['Metabolics_System']['Data'][:,code]
ind = ind[0:-1] - ind[1:]
bool_ind = ind != 0.

indShift = np.where(bool_ind)[0] # last index for each phase
indShift = np.concatenate((indShift, [len(ind)]))

for i in range(0,len(indShift)):
    T = result['Subject01']['Walking']['Metabolics_System']['Data'][indShift[i],0]
    timeM3 = T - (3*60)
    bool_time = result['Subject01']['Walking']['Metabolics_System']['Data'][:,0] < timeM3
    threeMin = indShift[i] - np.max(np.where(bool_time)[0])
    
    if i == 0:
        EE_ground[0:indShift[i]] = np.mean(EE[indShift[i]-threeMin:indShift[i]])
    elif i == len(indShift)-1:
        EE_ground[indShift[i-1]:] = np.mean(EE[-threeMin:])
    else:
        EE_ground[indShift[i-1]:indShift[i]+1] = np.mean(EE[indShift[i]-threeMin:indShift[i]])
